# DPE head reweighting — Kaggle

Run the H2 experiment on **precomputed raw ResNet-50 features**. This notebook uses the Python runner; it does not train Stage 0 or extract image features.

1. Enable **Internet** in Kaggle notebook settings. GPU is optional (the notebook uses it when available).
2. Add a Kaggle Dataset containing `metadata.csv`, `ckpt_last.pt`, `val_raw.npz`, `test_raw.npz`, and an audited `provenance.json` for **one track and seed**.
3. Edit the paths below, then run the cells in order. Start with standard seed 0. Repeat separately for seeds 1/2 and enhanced ERM*.

Feature NPZs contain exactly `ids` (Unicode filenames in metadata split order) and `features` (raw, unstandardized float arrays `[1199, 2048]` / `[5794, 2048]`). Provenance must identify the matching track, seed, checkpoint and validation-feature SHA-256 hashes, transforms, pretrained weights and source commit. The backbone must use final-epoch selection without validation/test tuning. See the [artifact contract](https://github.com/chautanphat/waterbirds-shortcut-mitigation/blob/experiment/dpe-head-reweighting/src/methods/dpe_head_reweighting/README.md#setup-and-input-artifacts).

Only validation A trains/selects heads; B fits weights; C selects aggregation. Test evaluation occurs in its own cell after selection is saved. Existing full-validation heads are not reused.

In [ ]:
# Install only missing dependencies and check out the tested implementation.
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

missing = [package for module, package in
           [("numpy", "numpy"), ("torch", "torch"), ("yaml", "pyyaml"), ("matplotlib", "matplotlib")]
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

REPO = Path("/kaggle/working/waterbirds-shortcut-mitigation")
CODE_REF = "10b25e2"  # Tested H2 runner; keep fixed throughout an experiment.
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", "experiment/dpe-head-reweighting",
                    "https://github.com/chautanphat/waterbirds-shortcut-mitigation.git", str(REPO)], check=True)
current = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short=7", "HEAD"], text=True).strip()
if current != CODE_REF:
    subprocess.run(["git", "-C", str(REPO), "checkout", CODE_REF], check=True)
os.chdir(REPO)
print("Implementation:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 1. Set your input paths

Change `INPUT_DIR` to your attached dataset folder. Adjust individual filenames if needed. The supplied provenance must match `TRACK` and `SEED`; do not relabel artifacts from another run. Use a new `RUN_NAME` for a different experiment. Re-running unchanged cells resumes existing stages without overwriting them.

In [ ]:
import json
import torch
import yaml

TRACK = "standard"  # "standard" or "enhanced"
SEED = 0            # 0, 1 or 2; requires a matching Stage-0 seed
RUN_NAME = f"h2-{TRACK}-seed{SEED}"
INPUT_DIR = Path("/kaggle/input/YOUR-DATASET-SLUG")

inputs = {
    "metadata": INPUT_DIR / "metadata.csv",
    "backbone": INPUT_DIR / "ckpt_last.pt",
    "val_features": INPUT_DIR / "val_raw.npz",
    "test_features": INPUT_DIR / "test_raw.npz",
    "provenance": INPUT_DIR / "provenance.json",
}
# Test features may be supplied later, before running the final test cell.
missing = [str(path) for key, path in inputs.items() if key != "test_features" and not path.is_file()]
if missing:
    raise FileNotFoundError("Attach the input artifacts and update INPUT_DIR:\n" + "\n".join(missing))

cfg = yaml.safe_load((REPO / f"configs/dpe_head_reweighting/waterbirds_{TRACK}.yaml").read_text())
cfg.update(track=TRACK, seed=SEED, device="cuda" if torch.cuda.is_available() else "cpu")
cfg.update({key: str(path) for key, path in inputs.items()})
RUN_DIR = Path("/kaggle/working") / RUN_NAME
cfg.update(experiment_dir=str(RUN_DIR / "experiment"),
           ensemble_path=str(RUN_DIR / "checkpoints/ensemble.pt"),
           results_dir=str(RUN_DIR / "results"))
CONFIG_PATH = RUN_DIR / "config.yaml"
RUN_DIR.mkdir(parents=True, exist_ok=True)
resolved_yaml = yaml.safe_dump(cfg, sort_keys=False)
if CONFIG_PATH.exists() and CONFIG_PATH.read_text() != resolved_yaml:
    raise ValueError("This RUN_NAME already has a different configuration. Choose a new RUN_NAME.")
CONFIG_PATH.write_text(resolved_yaml)

# These imports use the pinned repository's implementation.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.methods.dpe_head_reweighting.experiment import config, context, digest, get_ensemble, lock, read_json
resolved = config(CONFIG_PATH)

def run(action):
    print(f"Running {action}: {TRACK}, seed {SEED}, {cfg['device']}", flush=True)
    subprocess.run([sys.executable, str(REPO / "scripts/dpe_head_reweighting.py"),
                    action, "--config", str(CONFIG_PATH)], cwd=REPO, check=True)

print("Outputs:", RUN_DIR)
print("Device:", cfg["device"])
print("Stage-0 provenance:")
print(inputs["provenance"].read_text())

## 2. Prepare the fixed validation split

Verify artifact hashes and official split counts, then save the 60% A / 20% B / 20% C split (split seed 0). This does not require test features.

In [ ]:
manifest_path = Path(resolved["experiment_dir"]) / "manifest.json"
if manifest_path.exists():
    context(resolved)  # Reject changed inputs instead of silently reusing them.
    print("Reusing verified manifest.")
else:
    run("prepare")
manifest = read_json(manifest_path)
print("Subset sizes:", {role: len(rows) for role, rows in manifest["parts"].items()})

## 3. Train and export the 15 heads

Uses A for balanced training and epoch selection, 20 epochs per head. No test evaluation occurs here. CPU may take longer; the cell prints its completion when the artifact is saved.

In [ ]:
if Path(resolved["ensemble_path"]).exists():
    context(resolved)
    heads = get_ensemble(resolved)
    print("Reusing verified heads; selected epochs:", heads["selected_epochs"])
else:
    run("train-heads")

## 4. Fit weights on B and select on C

Compare uniform averaging with the 12 predeclared weighted candidates. The saved selection includes the chosen weights, all candidate metrics and artifact hashes. If uniform wins, retain that result.

In [ ]:
selection_path = Path(resolved["experiment_dir"]) / "selection.json"
if selection_path.exists():
    context(resolved)
    selection = read_json(selection_path)
    if selection["lock"] != lock(resolved):
        raise ValueError("Saved selection inputs changed; use a fresh experiment.")
    print("Reusing frozen selection.")
else:
    run("fit-select")
selection = read_json(selection_path)
print(json.dumps(selection["selected"], indent=2))

## 5. Final held-out evaluation

Run after selection is frozen. This evaluates the matched uniform control and selected aggregation on the same test predictions. Keep the declared grid and partition fixed after seeing these results.

In [ ]:
metrics_path = Path(resolved["results_dir"]) / "metrics.json"
if metrics_path.exists():
    context(resolved)
    result = read_json(metrics_path)
    if result["lock"] != lock(resolved) or result["test_features_hash"] != digest(inputs["test_features"]) or result["metadata_hash"] != digest(inputs["metadata"]):
        raise ValueError("Existing test result inputs changed; preserve the original experiment.")
    print("Displaying existing final evaluation.")
else:
    if not inputs["test_features"].is_file():
        raise FileNotFoundError(f"Attach the test raw-feature NPZ: {inputs['test_features']}")
    run("evaluate")
result = read_json(metrics_path)
for method in ("uniform", "selected"):
    m = result[method]
    print(f"{method:>8}: accuracy={m['overall']:.2%}, WGA={m['wga']:.2%}")
    print("          per-group accuracy:", m["groups"])
print("Paired changes (percentage points):", result["delta_pp"])
print("Effective head count:", result["effective_heads"])
from IPython.display import display, Image
plot = Path(resolved["results_dir"]) / "head_diagnostics.png"
display(Image(filename=str(plot)))

## 6. Download the run

The ZIP contains configuration, partition manifest, trained heads, frozen weights, results and diagnostic plots. It also includes per-head prediction caches when present. Download it from Kaggle's **Output** panel, or save a notebook version with outputs. No input checkpoint or raw features are copied into it.

Record this notebook version and Kaggle hardware with your run. One seed is preliminary; keep standard and enhanced results separate. The repository CLI can summarize each track after its three seed runs have been collected in the paths declared by a shared configuration.

In [ ]:
import shutil
cache_dir = REPO / "outputs/dpe_head_reweighting" / TRACK / f"seed{SEED}"
if cache_dir.exists():
    shutil.copytree(cache_dir, RUN_DIR / "predictions", dirs_exist_ok=True)
zip_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("Download:", zip_path)